# Bearing Health State Classification (Group-aware)

本 Notebook 包含完整流程：
1. 读取数据
2. EDA（类别分布、case 分布、关键特征可视化、原始波形抽样可视化）
3. 特征工程（结合转速、采样率、故障特征频率）
4. `GroupKFold(case_id)` 交叉验证（避免 case 泄漏）
5. 训练全量模型并生成提交文件


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

# Auto-detect input data dir across local + competition runtime
candidates = [
    Path('./dataset/public'),
    Path('/root/setup/solution/dataset/public'),
    Path('./public'),
    Path('.'),
    Path('/Users/songling/Desktop/public'),
]

DATA_DIR = None
for p in candidates:
    if (p / 'train.csv').exists() and (p / 'test.csv').exists() and (p / 'signals').exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    for p in Path('.').rglob('train.csv'):
        parent = p.parent
        if (parent / 'test.csv').exists() and (parent / 'signals').exists():
            DATA_DIR = parent
            break

if DATA_DIR is None:
    raise FileNotFoundError('Cannot find train.csv/test.csv/signals in runtime environment')

# Auto-detect output dir (competition platform usually uses /working)
out_candidates = [
    Path('./working'),
    Path('/root/setup/solution/working'),
    Path('.'),
]
OUTPUT_DIR = None
for p in out_candidates:
    if p.exists() and p.is_dir():
        OUTPUT_DIR = p
        break
if OUTPUT_DIR is None:
    OUTPUT_DIR = Path('.')

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SIGNALS_DIR = DATA_DIR / 'signals'
SAMPLE_SUB_PATH = DATA_DIR / 'sample_submission.csv'

print('DATA_DIR:', DATA_DIR.resolve())
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())
print('train exists:', TRAIN_PATH.exists())
print('test exists:', TEST_PATH.exists())
print('signals exists:', SIGNALS_DIR.exists())
print('HAS_SEABORN:', HAS_SEABORN)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print()
print('train columns:')
print(train_df.columns.tolist())


In [ ]:
train_df.head(3), test_df.head(3)


## EDA

In [ ]:
print('Train case_id count:', train_df['case_id'].nunique())
print('Test  case_id count:', test_df['case_id'].nunique())
print('Case overlap:', len(set(train_df['case_id']).intersection(set(test_df['case_id']))))

print()
print('Target distribution:')
print(train_df['Bearing_health_state'].value_counts().sort_index())
print()
print('Target ratio:')
print((train_df['Bearing_health_state'].value_counts(normalize=True).sort_index()*100).round(2).astype(str)+'%')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

vc = train_df['Bearing_health_state'].value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values)
axes[0].set_title('Target Distribution (Train)')

asset_vc = train_df['asset_type'].value_counts()
axes[1].bar(asset_vc.index.astype(str), asset_vc.values)
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_title('Asset Type Distribution (Train)')

sensor_vc = train_df['sensor_position'].value_counts()
axes[2].bar(sensor_vc.index.astype(str), sensor_vc.values)
axes[2].set_title('Sensor Position Distribution (Train)')

plt.tight_layout()
plt.show()


In [ ]:
num_cols_for_eda = [
    'rms','peak_to_peak','kurtosis','fft_peak_freq_norm',
    'spectral_centroid','spectral_bandwidth','band_energy_high','rpm'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()
for i, c in enumerate(num_cols_for_eda):
    groups = [train_df.loc[train_df['Bearing_health_state']==k, c].dropna().values for k in sorted(train_df['Bearing_health_state'].unique())]
    axes[i].boxplot(groups, tick_labels=[str(k) for k in sorted(train_df['Bearing_health_state'].unique())], showfliers=False)
    axes[i].set_title(c)
    axes[i].set_xlabel('Bearing_health_state')
plt.tight_layout()
plt.show()


In [ ]:
# 原始信号抽样可视化（每个类别随机看1条）
def load_signal(mid):
    path = os.path.join(SIGNALS_DIR, f'{int(mid):05d}.npy')
    return np.load(path)

sample_rows = []
for cls in sorted(train_df['Bearing_health_state'].unique()):
    sample_rows.append(train_df[train_df['Bearing_health_state'] == cls].sample(1, random_state=SEED))
sample_rows = pd.concat(sample_rows, axis=0).reset_index(drop=True)

fig, axes = plt.subplots(len(sample_rows), 2, figsize=(14, 3*len(sample_rows)))
if len(sample_rows) == 1:
    axes = np.array([axes])

for i, row in sample_rows.iterrows():
    sig = load_signal(row['measurement_id'])
    sr = float(row['sampling_rate'])

    t = np.arange(len(sig)) / sr
    axes[i,0].plot(t[:3000], sig[:3000], lw=1)
    axes[i,0].set_title(f"class={int(row['Bearing_health_state'])}, id={int(row['measurement_id'])} (time)")
    axes[i,0].set_xlabel('sec')

    spec = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(len(sig), d=1.0/sr)
    axes[i,1].plot(freqs[1:2000], spec[1:2000], lw=1)
    axes[i,1].set_title(f"class={int(row['Bearing_health_state'])}, id={int(row['measurement_id'])} (spectrum)")
    axes[i,1].set_xlabel('Hz')

plt.tight_layout()
plt.show()


## Feature Engineering

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-9

    out['rot_hz'] = out['rpm'] / 60.0
    out['nyquist_hz'] = out['sampling_rate'] / 2.0

    out['ftf_hz'] = out['ftf_multiple'] * out['rot_hz']
    out['bpf_hz'] = out['bpf_multiple'] * out['rot_hz']
    out['bpfo_hz'] = out['bpfo_multiple'] * out['rot_hz']
    out['bpfi_hz'] = out['bpfi_multiple'] * out['rot_hz']

    out['ftf_norm'] = out['ftf_hz'] / (out['nyquist_hz'] + eps)
    out['bpf_norm'] = out['bpf_hz'] / (out['nyquist_hz'] + eps)
    out['bpfo_norm'] = out['bpfo_hz'] / (out['nyquist_hz'] + eps)
    out['bpfi_norm'] = out['bpfi_hz'] / (out['nyquist_hz'] + eps)

    out['fft_peak_hz'] = out['fft_peak_freq_norm'] * out['nyquist_hz']

    out['delta_ftf'] = np.abs(out['fft_peak_hz'] - out['ftf_hz'])
    out['delta_bpf'] = np.abs(out['fft_peak_hz'] - out['bpf_hz'])
    out['delta_bpfo'] = np.abs(out['fft_peak_hz'] - out['bpfo_hz'])
    out['delta_bpfi'] = np.abs(out['fft_peak_hz'] - out['bpfi_hz'])
    out['min_fault_delta'] = out[['delta_ftf','delta_bpf','delta_bpfo','delta_bpfi']].min(axis=1)

    out['band_energy_sum'] = (
        out['band_energy_low'] + out['band_energy_mid_low'] +
        out['band_energy_mid_high'] + out['band_energy_high']
    )
    out['band_low_ratio'] = out['band_energy_low'] / (np.abs(out['band_energy_sum']) + eps)
    out['band_mid_low_ratio'] = out['band_energy_mid_low'] / (np.abs(out['band_energy_sum']) + eps)
    out['band_mid_high_ratio'] = out['band_energy_mid_high'] / (np.abs(out['band_energy_sum']) + eps)
    out['band_high_ratio'] = out['band_energy_high'] / (np.abs(out['band_energy_sum']) + eps)

    out['rpm_is_zero'] = (out['rpm'] == 0).astype(int)
    out['signal_seconds'] = out['signal_length'] / (out['sampling_rate'] + eps)

    out['peak_rms_ratio2'] = out['peak'] / (out['rms'] + eps)
    out['std_rms_ratio'] = out['std'] / (out['rms'] + eps)

    return out


## Modeling (GroupKFold by case_id)

In [ ]:
full_df = pd.concat([train_df.drop(columns=['Bearing_health_state']), test_df], axis=0, ignore_index=True)
full_df = add_features(full_df)

cat_cols = ['sensor_position', 'asset_type', 'bearing_model', 'fault_origin', 'unit']
for c in cat_cols:
    full_df[c] = full_df[c].astype('category')

train_X = full_df.iloc[:len(train_df)].copy()
test_X = full_df.iloc[len(train_df):].copy()
y = train_df['Bearing_health_state'].astype(int).values
groups = train_df['case_id'].values

feature_drop_cols = ['measurement_id', 'case_id']
X_cols = [c for c in train_X.columns if c not in feature_drop_cols]

train_X = train_X[X_cols]
test_X = test_X[X_cols]

print('feature count:', len(X_cols))
print('categorical cols used:', [c for c in cat_cols if c in train_X.columns])


In [ ]:
from sklearn.utils import resample

def oversample_minority(X: pd.DataFrame, y: np.ndarray, ratio_to_major=0.35, random_state=42):
    df = X.copy()
    df['_y_'] = y

    counts = df['_y_'].value_counts()
    maj_class = counts.idxmax()
    maj_n = counts.max()

    target_n = {cls: max(cnt, int(maj_n * ratio_to_major)) for cls, cnt in counts.items()}
    target_n[maj_class] = maj_n

    parts = []
    rng = np.random.RandomState(random_state)
    for cls in sorted(counts.index):
        part = df[df['_y_'] == cls]
        if len(part) < target_n[cls]:
            part_up = resample(
                part,
                replace=True,
                n_samples=target_n[cls],
                random_state=int(rng.randint(0, 1_000_000))
            )
            parts.append(part_up)
        else:
            parts.append(part)

    out = pd.concat(parts, axis=0).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    y_out = out.pop('_y_').values
    return out, y_out


In [ ]:
try:
    import lightgbm as lgb
    USE_LGB = True
except Exception as e:
    print('LightGBM import failed:', e)
    USE_LGB = False

print('USE_LGB =', USE_LGB)


In [ ]:
from sklearn.ensemble import RandomForestClassifier


def train_cv(train_X, y, groups, n_splits=5, seed=42):
    gkf = GroupKFold(n_splits=n_splits)

    oof_proba_cfg1 = np.zeros((len(train_X), 4), dtype=float)
    oof_proba_cfg2 = np.zeros((len(train_X), 4), dtype=float)

    models_cfg1 = []
    models_cfg2 = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(train_X, y, groups), 1):
        X_tr, y_tr = train_X.iloc[tr_idx].copy(), y[tr_idx]
        X_va, y_va = train_X.iloc[va_idx].copy(), y[va_idx]

        X_tr_os, y_tr_os = oversample_minority(X_tr, y_tr, ratio_to_major=0.35, random_state=seed+fold)

        use_lgb_this_fold = USE_LGB and (len(np.unique(y_tr)) == 4)

        if use_lgb_this_fold:
            model1 = lgb.LGBMClassifier(
                objective='multiclass', num_class=4,
                learning_rate=0.03, n_estimators=1200,
                num_leaves=31, min_child_samples=20,
                subsample=0.9, colsample_bytree=0.85,
                reg_alpha=0.2, reg_lambda=0.2,
                random_state=seed+fold,
                class_weight='balanced', verbosity=-1
            )
            model1.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric='multi_logloss',
                callbacks=[lgb.early_stopping(120, verbose=False)]
            )

            model2 = lgb.LGBMClassifier(
                objective='multiclass', num_class=4,
                learning_rate=0.03, n_estimators=1400,
                num_leaves=63, min_child_samples=15,
                subsample=0.9, colsample_bytree=0.9,
                reg_alpha=0.1, reg_lambda=0.1,
                random_state=seed+100+fold, verbosity=-1
            )
            model2.fit(
                X_tr_os, y_tr_os,
                eval_set=[(X_va, y_va)],
                eval_metric='multi_logloss',
                callbacks=[lgb.early_stopping(120, verbose=False)]
            )

            p1 = model1.predict_proba(X_va)
            p2 = model2.predict_proba(X_va)
        else:
            tr_d = pd.get_dummies(X_tr)
            va_d = pd.get_dummies(X_va)
            tr_os_d = pd.get_dummies(X_tr_os)

            model1 = RandomForestClassifier(
                n_estimators=600,
                random_state=seed+fold,
                class_weight='balanced_subsample',
                n_jobs=-1
            )
            model1.fit(tr_d, y_tr)

            model2 = RandomForestClassifier(
                n_estimators=700,
                random_state=seed+100+fold,
                class_weight='balanced_subsample',
                n_jobs=-1
            )
            model2.fit(tr_os_d, y_tr_os)

            p1_raw = model1.predict_proba(va_d.reindex(columns=model1.feature_names_in_, fill_value=0))
            p2_raw = model2.predict_proba(va_d.reindex(columns=model2.feature_names_in_, fill_value=0))

            p1 = np.zeros((len(X_va), 4), dtype=float)
            p2 = np.zeros((len(X_va), 4), dtype=float)
            p1[:, model1.classes_.astype(int)] = p1_raw
            p2[:, model2.classes_.astype(int)] = p2_raw

        p_ens = 0.55 * p1 + 0.45 * p2
        oof_proba_cfg1[va_idx] = p1
        oof_proba_cfg2[va_idx] = p2

        s1 = f1_score(y_va, p1.argmax(axis=1), average='macro')
        s2 = f1_score(y_va, p2.argmax(axis=1), average='macro')
        se = f1_score(y_va, p_ens.argmax(axis=1), average='macro')
        print(f'Fold {fold}: cfg1={s1:.5f} | cfg2={s2:.5f} | ens={se:.5f}')

        models_cfg1.append(model1)
        models_cfg2.append(model2)

    cv1 = f1_score(y, oof_proba_cfg1.argmax(axis=1), average='macro')
    cv2 = f1_score(y, oof_proba_cfg2.argmax(axis=1), average='macro')
    cve = f1_score(y, (0.55*oof_proba_cfg1+0.45*oof_proba_cfg2).argmax(axis=1), average='macro')

    print()
    print('OOF Macro F1:')
    print(f'cfg1: {cv1:.6f}')
    print(f'cfg2: {cv2:.6f}')
    print(f'ens : {cve:.6f}')

    return {
        'models_cfg1': models_cfg1,
        'models_cfg2': models_cfg2,
        'oof_p1': oof_proba_cfg1,
        'oof_p2': oof_proba_cfg2,
        'cv1': cv1,
        'cv2': cv2,
        'cve': cve,
    }


In [ ]:
cv_result = train_cv(train_X, y, groups, n_splits=5, seed=SEED)


In [ ]:
oof_pred = (0.55*cv_result['oof_p1'] + 0.45*cv_result['oof_p2']).argmax(axis=1)
print(classification_report(y, oof_pred, digits=4))

cm = confusion_matrix(y, oof_pred, labels=[0,1,2,3])
plt.figure(figsize=(5,4))
if HAS_SEABORN:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1,2,3], yticklabels=[0,1,2,3])
else:
    plt.imshow(cm, cmap='Blues')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')
    plt.xticks([0,1,2,3], [0,1,2,3])
    plt.yticks([0,1,2,3], [0,1,2,3])
    plt.colorbar()
plt.xlabel('Pred')
plt.ylabel('True')
plt.title('OOF Confusion Matrix')
plt.tight_layout()
plt.show()


## Train Full Data & Predict Test

In [ ]:
def fit_full_models(train_X, y, seed=42):
    if USE_LGB:
        import lightgbm as lgb
        m1 = lgb.LGBMClassifier(
            objective='multiclass', num_class=4,
            learning_rate=0.03, n_estimators=1200,
            num_leaves=31, min_child_samples=20,
            subsample=0.9, colsample_bytree=0.85,
            reg_alpha=0.2, reg_lambda=0.2,
            class_weight='balanced',
            random_state=seed, verbosity=-1
        )
        m1.fit(train_X, y)

        X_os, y_os = oversample_minority(train_X, y, ratio_to_major=0.35, random_state=seed)
        m2 = lgb.LGBMClassifier(
            objective='multiclass', num_class=4,
            learning_rate=0.03, n_estimators=1400,
            num_leaves=63, min_child_samples=15,
            subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.1, reg_lambda=0.1,
            random_state=seed+100, verbosity=-1
        )
        m2.fit(X_os, y_os)
        return m1, m2

    tr_d = pd.get_dummies(train_X)
    m1 = RandomForestClassifier(n_estimators=600, random_state=seed, class_weight='balanced_subsample', n_jobs=-1)
    m1.fit(tr_d, y)

    X_os, y_os = oversample_minority(train_X, y, ratio_to_major=0.35, random_state=seed)
    tr_os_d = pd.get_dummies(X_os)
    m2 = RandomForestClassifier(n_estimators=700, random_state=seed+100, class_weight='balanced_subsample', n_jobs=-1)
    m2.fit(tr_os_d, y_os)
    return m1, m2

full_m1, full_m2 = fit_full_models(train_X, y, seed=SEED)
print('full models trained.')


In [ ]:
if USE_LGB:
    test_p1 = full_m1.predict_proba(test_X)
    test_p2 = full_m2.predict_proba(test_X)
else:
    test_d = pd.get_dummies(test_X)
    test_p1 = full_m1.predict_proba(test_d.reindex(columns=full_m1.feature_names_in_, fill_value=0))
    test_p2 = full_m2.predict_proba(test_d.reindex(columns=full_m2.feature_names_in_, fill_value=0))

test_pred = (0.55 * test_p1 + 0.45 * test_p2).argmax(axis=1).astype(int)

sub = pd.DataFrame({
    'measurement_id': test_df['measurement_id'].astype(int),
    'Bearing_health_state': test_pred
})

sub_path = OUTPUT_DIR / 'submission.csv'
sub.to_csv(sub_path, index=False)

print('Saved:', sub_path.resolve())
print(sub.head())
print('rows:', len(sub), 'unique measurement_id:', sub['measurement_id'].nunique())
print('label distribution:')
print(sub['Bearing_health_state'].value_counts().sort_index())


## Notes

- 关键点是使用 `GroupKFold(case_id)`，模拟“新安装场景”泛化。
- 类别极不平衡（尤其 class 2 很少），因此使用“原始分布模型 + 轻度过采样模型”集成。
- 可继续提分方向：
  - 从 `signals/*.npy` 批量提取更多包络/频带特征；
  - 按 `sensor_position` 或 `asset_type` 做分层建模并融合；
  - 在 GroupKFold 下做更系统化参数搜索。
